<a href="https://colab.research.google.com/drive/1JmF5MaE5cutvTB3LkwGlrr5nBc_c1dc3?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>

<div style="text-align:center;">
<a href="https://lexsi-labs.github.io/CuratorKIT/latest/"><img src="https://raw.githubusercontent.com/Lexsi-Labs/CuratorKIT/refs/heads/main/docs/assets/logo.png" width="450"></a>

<a href="https://lexsi-labs.github.io/CuratorKIT/latest/"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/docs.png" width="150"></a>

</div>

<i>Star us on <a href="https://github.com/Lexsi-Labs/CuratorKIT">Github</a> </i>

To run this, press "*Runtime*" and press "*Run all*" on a Google Colab instance!

<div style="text-align:center;">
<a href="https://arxiv.org/abs/2606.21631"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/lexsilogo.png" width="200"></a>
</div>
</center>




# Data ingestion & cleaning

**Pattern:** Single dataset with auto-detection, no custom preprocessing, no resampling.

Simplest possible pipeline. Reads one dataset, auto-detects the format,
runs the full dedup → clean → filter chain, and exports.

### When to use this vs. multi-reader
- **Single reader:** You have one clean dataset and just want dedup + cleaning + export
- **Multi-reader:** You're merging heterogeneous sources and need normalization + resampling

### Pipeline flow
```
tatsu-lab/alpaca  →  auto-detect format (alpaca)  →  SchemaGate
                    →  ExactDedup → MinHashDedup   →  TextCleaner
                    →  DiversityGate               →  AlpacaExporter
```

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# 0. Setup — clone + install CuratorKIT (run once, then comment out)
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

repo_dir = Path.home() / "CuratorKIT"
if not repo_dir.exists():
    !git clone https://github.com/Lexsi-Labs/CuratorKIT.git {repo_dir}

!pip install -e "{repo_dir}[generation,embedding,hf,parquet]" -q

print(f"CuratorKIT installed from {repo_dir}")



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

## 1. Run pipeline

In [2]:
from curatorkit import Curator, CuratorConfig

config = CuratorConfig(
    # ── Single source — auto-detected as alpaca format ────────────────────
    dataset="tatsu-lab/alpaca",
    split="train",
    max_samples=5000,

    # ── Schema gate ───────────────────────────────────────────────────────
    min_tokens=10,
    max_tokens=1024,

    # ── Dedup (exact + fuzzy) ─────────────────────────────────────────────
    dedup="minhash",
    minhash_threshold=0.85,

    # ── Text cleaning ─────────────────────────────────────────────────────
    clean=True,

    # ── Semantic diversity filter ─────────────────────────────────────────
    diversity_threshold=0.92,
    embedding_device="cuda",

    # ── Export ────────────────────────────────────────────────────────────
    export_formats=["alpaca", "sharegpt"],
    output_dir="output/data_ingestion_single_reader",
)

curator = Curator(config)
curator.dry_run()

print("\n=== Running ===")
result = curator.run()
result.print_summary()


=== Pipeline.dry_run — 9 step(s) ===
Output dir: output/data_ingestion_single_reader

   1. [reader    ] HuggingFaceReader
   2. [gate      ] SchemaGate  (cfg=ffff512734781804)
   3. [normalizer] ExactDeduplicator  (cfg=98ec237522ba1e05)
   4. [normalizer] MinHashDeduplicator  (cfg=9315210001ce73f7)
   5. [normalizer] TextCleaner  (cfg=2176e3c3449cf50b)
   6. [gate      ] DiversityGate  (cfg=7ec497e4b545234e)
   7. [normalizer] MaxSamplesTruncator
   8. [exporter  ] AlpacaExporter
   9. [exporter  ] ShareGPTExporter

Always emitted: manifest.json, dataset_card.md, rejected.jsonl, checksums.txt
=== END Pipeline.dry_run ===


=== Running ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

DiversityGate: 100%|██████████| 50548/50548 [15:59<00:00, 52.69sample/s]


────────────────────────────────────────────
  passed   :    5,000
  rejected :    2,199
  time     :  1618.4s
  output   : output/data_ingestion_single_reader
────────────────────────────────────────────


## 2. Inspect

In [3]:
from collections import Counter

print(f"Passed: {len(result.passed):,}  |  Rejected: {len(result.rejected):,}")

# ── Rejection breakdown ───────────────────────────────────────────────────
if result.rejected:
    reasons = Counter(r.rejection_reason for r in result.rejected)
    print("\nRejection reasons:")
    for reason, count in reasons.most_common():
        print(f"  {count:>4}  {reason}")

# ── Sample preview ────────────────────────────────────────────────────────
print("\nSamples:")
for i, s in enumerate(result.passed[:3]):
    print(f"\n── {i+1} ({s.task_type}) ──")
    print(f"  Q: {s.instruction[:120]}")
    print(f"  A: {s.output[:120]}")

Passed: 5,000  |  Rejected: 2,199

Rejection reasons:
   645  below_min_tokens:9
   350  below_min_tokens:8
   221  below_min_tokens:7
   124  below_min_tokens:6
    82  below_min_tokens:5
    37  diversity_gate:too_similar:0.922
    34  diversity_gate:too_similar:0.929
    32  diversity_gate:too_similar:0.925
    28  missing_field:output
    28  diversity_gate:too_similar:0.923
    27  diversity_gate:too_similar:0.921
    26  diversity_gate:too_similar:0.924
    25  diversity_gate:too_similar:0.931
    24  diversity_gate:too_similar:0.926
    24  diversity_gate:too_similar:0.936
    23  diversity_gate:too_similar:0.927
    22  diversity_gate:too_similar:0.941
    22  diversity_gate:too_similar:0.933
    21  diversity_gate:too_similar:0.939
    21  diversity_gate:too_similar:0.935
    21  diversity_gate:too_similar:0.937
    20  diversity_gate:too_similar:0.940
    20  diversity_gate:too_similar:0.943
    19  diversity_gate:too_similar:0.928
    18  diversity_gate:too_similar:0.948
   

In [4]:
# ── Stage counts ──────────────────────────────────────────────────────────
print("Pipeline stages:")
for step, counts in result.stage_counts.items():
    parts = [f"{k}={v}" for k, v in counts.items()]
    print(f"  {step}: {', '.join(parts)}")

Pipeline stages:
  HuggingFaceReader: output_count=52002, rejected_count=0
  SchemaGate: input_count=52002, output_count=50552, probe_recovered=0, rejected_count=1450
  ExactDeduplicator: input_count=50552, output_count=50552
  MinHashDeduplicator: input_count=50552, output_count=50548
  TextCleaner: input_count=50548, output_count=50548
  DiversityGate: input_count=50548, output_count=49799, probe_recovered=0, rejected_count=749
  MaxSamplesTruncator: input_count=49799, output_count=5000
  AlpacaExporter: exported_count=5000
  ShareGPTExporter: exported_count=5000


## 3. Single-reader vs Multi-reader

| | Single-Reader | Multi-Reader |
|---|---|---|
| **Dataset** | 1 source | 2+ sources |
| **Config** | `dataset="name"` | `dataset=[{"name": "...", "max_samples": N}, ...]` |
| **Preprocessing** | Auto-detect format | Custom `preprocessing_fn` per reader |
| **Resampling** | Not needed | `resample=True` by source_dataset |
| **Use case** | Clean single dataset | Merging heterogeneous sources |

The single-reader path relies entirely on CuratorKIT's auto-detection (Layer 1-3 of FormatDetector).
The multi-reader path uses `preprocessing_fn` to normalize schemas before auto-detection runs.